# Backbone Experiment: convnext_base

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device: cuda
GPU:    NVIDIA RTX PRO 4000 Blackwell
VRAM:   25.2 GB


## Shared Config

In [2]:
# Only line to change
BACKBONE = "convnext_base"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [3]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: cuda


Training spatial-only baseline (convnext_base) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.1203 | val_acc=97.0%


Epoch   2/50 | train_loss=0.0678 | val_acc=98.1%


Epoch   3/50 | train_loss=0.0505 | val_acc=98.2%


...

Epoch  46/50 | train_loss=0.0004 | val_acc=98.7%


Epoch  47/50 | train_loss=0.0002 | val_acc=98.7%


Epoch  48/50 | train_loss=0.0003 | val_acc=98.8%


Epoch  49/50 | train_loss=0.0002 | val_acc=98.8%


Epoch  50/50 | train_loss=0.0002 | val_acc=98.7%

Spatial-only results (convnext_base):
  Accuracy: 98.9%
  AUC-ROC:  0.998
  F1:       0.989
Results saved → ./results/results.csv  (convnext_base_spatial_only, acc=0.9892)
Results saved to ./results/results.csv

Spatial-only floor: 98.9%
All subsequent delta values are relative to this number.


## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [3]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

Running: convnext_base_joint_only
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch 1:   0%|          | 0/1329 [00:00<?, ?batch/s]/home/peter-kabati/Desktop/DeepLearning(overall)/asfr/utils/fft_utils.py:37: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at /pytorch/aten/src/ATen/EmptyTensor.cpp:54.)
  spectrum = torch.fft.fft2(image_tensor)


Epoch   1/50 | train_loss=0.3600 | val_acc=97.5% | val_auc=0.997 | val_f1=0.975
  grad norms — freq=0.8502 | spatial=0.5235
  -> Saved best val_acc=97.5%


Epoch   2/50 | train_loss=0.2565 | val_acc=97.9% | val_auc=0.998 | val_f1=0.979
  -> Saved best val_acc=97.9%


Epoch   3/50 | train_loss=0.2242 | val_acc=97.9% | val_auc=0.998 | val_f1=0.979


...

Epoch  47/50 | train_loss=0.0830 | val_acc=98.9% | val_auc=0.998 | val_f1=0.989
  -> Saved best val_acc=98.9%


Epoch  48/50 | train_loss=0.0821 | val_acc=98.9% | val_auc=0.998 | val_f1=0.989


Epoch  49/50 | train_loss=0.0827 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  50/50 | train_loss=0.0815 | val_acc=98.9% | val_auc=0.998 | val_f1=0.989
Results saved → ./results/results.csv  (convnext_base_joint_only, acc=0.9886)

Training complete. Best val accuracy: 98.9%


0.9888

In [5]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_joint_only.pt

EVALUATION — convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
  Joint accuracy:     98.9%
  AUC-ROC:            0.999
  F1:                 0.989
  Spatial-only:       98.8%
  Freq-only:          92.5%
  Delta (Δ):          +0.0%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_joint_only, acc=0.9886)


## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch — b is the frequency weight.
If b drops below 0.1 early in training the frequency branch is being ignored.


In [6]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

Running: convnext_base_scalar
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.3657 | val_acc=97.4% | val_auc=0.997 | val_f1=0.974
  scalars — spatial=0.498 freq=0.502
  grad norms — freq=0.9967 | spatial=0.0787
  -> Saved best val_acc=97.4%


Epoch   2/50 | train_loss=0.2579 | val_acc=98.3% | val_auc=0.998 | val_f1=0.983
  scalars — spatial=0.496 freq=0.504
  -> Saved best val_acc=98.3%


Epoch   3/50 | train_loss=0.2244 | val_acc=97.9% | val_auc=0.998 | val_f1=0.979
  scalars — spatial=0.495 freq=0.505


Epoch   4/50 | train_loss=0.1999 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  scalars — spatial=0.494 freq=0.506
  -> Saved best val_acc=98.4%


Epoch   5/50 | train_loss=0.1835 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  scalars — spatial=0.493 freq=0.507


KeyboardInterrupt: 

In [ ]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"../checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [ ]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

In [ ]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"../checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3 —
essentially teaching itself what the frequency branch provides.


In [ ]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

In [ ]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"../checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

## Summary: convnext_base
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [ ]:
df = pd.read_csv("../results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
